# Overview


# Model Architectures

Diamantini et al. (2024) compare several neural-network architectures for KPI forecasting. Because full hyperparameter details are not fully documented in the paper, this notebook uses explicit equivalent baselines:

- **Transformer** — stacked encoder blocks with multi-head attention.
- **LSTM** — stacked recurrent layers.
- **MLP** — dense feed-forward network.
- **RNN** — stacked SimpleRNN layers.
- **CNN** — stacked Conv1D blocks.

For stability on raw `Sales`, models use output-layer bias initialized to `mean(y_train)`, per-model learning rates, gradient clipping, and `TerminateOnNaN`.


# Reference-Based Exploration

Exploration follows the Diamantini et al. (2024) Rossmann non-normalized forecasting setup. Preprocessing loads and merges `train.csv` + `store.csv`, engineers promo-related features, applies a chronological 80/20 split, and scales input features only (target remains raw). Modeling trains Transformer, LSTM, MLP, RNN, and CNN, then compares reproduction metrics against verified paper reference values.


## Preprocessing

Runtime setup, paper reference validation, dataset loading/validation, exploratory data analysis, and modeling data preparation.

The setup cell detects compute devices dynamically: **GPU is used when available**, otherwise training falls back to **CPU** with explicit log messages. Dataset paths are resolved from the project root (`data/raw/rossmann`). Compute device detection uses GPU when available, otherwise CPU.


In [1]:
import sys
import os
import random
import warnings
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).resolve()
if PROJECT_ROOT.name in {'explore', 'notebooks', 'Notebooks-to-transfers'}:
    PROJECT_ROOT = PROJECT_ROOT.parent
    if PROJECT_ROOT.name == 'notebooks':
        PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)


def configure_compute_device() -> dict:
    """Detect GPU/CPU dynamically and print a clear training environment summary."""
    physical_gpus = tf.config.list_physical_devices('GPU')
    logical_gpus = tf.config.list_logical_devices('GPU')

    for gpu in physical_gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError as exc:
            print(f'[WARN] Could not enable GPU memory growth: {exc}')

    gpu_available = len(logical_gpus) > 0
    if gpu_available:
        compute_device = 'GPU'
        device_name = physical_gpus[0].name
        try:
            details = tf.config.experimental.get_device_details(physical_gpus[0])
            device_name = details.get('device_name', device_name)
        except Exception:
            pass
        training_strategy = (
            tf.distribute.MirroredStrategy()
            if len(logical_gpus) > 1
            else tf.distribute.get_strategy()
        )
        device_message = '[INFO] GPU detected — deep learning training will use GPU acceleration.'
    else:
        compute_device = 'CPU'
        logical_cpus = tf.config.list_logical_devices('CPU')
        device_name = logical_cpus[0].name if logical_cpus else '/CPU:0'
        training_strategy = tf.distribute.get_strategy()
        device_message = '[WARN] No GPU detected — deep learning training will run on CPU (slower).'

    # Keep float32: raw Sales target + MSE can overflow with mixed_float16.
    keras.mixed_precision.set_global_policy('float32')

    print('=' * 60)
    print('Compute environment')
    print('=' * 60)
    print(f'Project root       : {PROJECT_ROOT}')
    print(f'TensorFlow version : {tf.__version__}')
    print(device_message)
    print(f'Compute device     : {compute_device}')
    print(f'Device name        : {device_name}')
    print(f'Physical GPUs      : {physical_gpus}')
    print(f'Logical GPUs       : {logical_gpus}')
    print(f'Training strategy  : {type(training_strategy).__name__}')
    print(f'Mixed precision    : {keras.mixed_precision.global_policy()}')
    print('=' * 60)

    return {
        'gpu_available': gpu_available,
        'compute_device': compute_device,
        'device_name': device_name,
        'training_strategy': training_strategy,
        'physical_gpus': physical_gpus,
        'logical_gpus': logical_gpus,
    }


compute = configure_compute_device()

GPU_AVAILABLE = compute['gpu_available']
COMPUTE_DEVICE = compute['compute_device']
DEVICE_NAME = compute['device_name']
TRAINING_STRATEGY = compute['training_strategy']
GPU_DEVICES = compute['physical_gpus']
LOGICAL_GPUS = compute['logical_gpus']

print('All imports successful.')


Compute environment
Project root       : D:\Work-Env\ITEC\forecasting-medicine-public
TensorFlow version : 2.20.0
[WARN] No GPU detected — deep learning training will run on CPU (slower).
Compute device     : CPU
Device name        : /device:CPU:0
Physical GPUs      : []
Logical GPUs       : []
Training strategy  : _DefaultDistributionStrategy
Mixed precision    : <DTypePolicy "float32">
All imports successful.


In [2]:
# Rossmann dataset location
DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'rossmann'
TRAIN_PATH = DATA_DIR / 'train.csv'
STORE_PATH = DATA_DIR / 'store.csv'

if not TRAIN_PATH.exists():
    raise FileNotFoundError(f'train.csv not found: {TRAIN_PATH}')
if not STORE_PATH.exists():
    raise FileNotFoundError(f'store.csv not found: {STORE_PATH}')

print(f'Data directory: {DATA_DIR}')
print(f'Train path    : {TRAIN_PATH}')
print(f'Store path    : {STORE_PATH}')


Data directory: D:\Work-Env\ITEC\forecasting-medicine-public\data\raw\rossmann
Train path    : D:\Work-Env\ITEC\forecasting-medicine-public\data\raw\rossmann\train.csv
Store path    : D:\Work-Env\ITEC\forecasting-medicine-public\data\raw\rossmann\store.csv


### Paper Reference Validation

The initial task extraction showed some RMSE/MAE values misaligned with the original paper table. This notebook keeps two sources:

- `SOURCE_EXTRACTED_REFERENCE` — values from the initial extraction/task description.
- `PAPER_REFERENCE` — verified values used for final comparison.

Note: Table 1 in the paper uses column order **MAE, MSE, RMSE, R Squared**. With this order, some initially extracted RMSE values actually belong to other models/metrics.


In [3]:
SOURCE_EXTRACTED_REFERENCE = {
    'Transformer': {'rmse': 1283.77, 'mse': 1475395.99, 'r2': 0.96, 'mae': 800.01},
    'LSTM': {'rmse': 1394.37, 'mse': 1646816.85, 'r2': 0.91, 'mae': 821.29},
    'MLP': {'rmse': 1932.60, 'mse': 1750258.53, 'r2': 0.89, 'mae': 1255.74},
    'RNN': {'rmse': np.nan, 'mse': 1945628.38, 'r2': 0.88, 'mae': 458.91},
    'CNN': {'rmse': np.nan, 'mse': 3734871.20, 'r2': 0.74, 'mae': np.nan},
}

PAPER_REFERENCE = {
    'MLP': {'mae': 835.54, 'mse': 1750258.53, 'rmse': 1322.57, 'r2': 0.89},
    'LSTM': {'mae': 800.01, 'mse': 1646816.85, 'rmse': 1283.77, 'r2': 0.91},
    'CNN': {'mae': 1255.74, 'mse': 3734871.20, 'rmse': 1932.60, 'r2': 0.74},
    'RNN': {'mae': 821.29, 'mse': 1945628.38, 'rmse': 1394.37, 'r2': 0.88},
    'Transformer': {'mae': 760.48, 'mse': 1475395.99, 'rmse': 1214.00, 'r2': 0.96},
}

source_df = pd.DataFrame(SOURCE_EXTRACTED_REFERENCE).T.add_prefix('source_')
paper_df_reference = pd.DataFrame(PAPER_REFERENCE).T.add_prefix('paper_')
paper_verification = source_df.join(paper_df_reference, how='outer')

for metric in ['rmse', 'mse', 'r2', 'mae']:
    paper_verification[f'{metric}_mismatch'] = ~np.isclose(
        paper_verification[f'source_{metric}'].astype(float),
        paper_verification[f'paper_{metric}'].astype(float),
        equal_nan=True,
    )

paper_verification


,source_rmse,source_mse,source_r2,source_mae,paper_mae,paper_mse,paper_rmse,paper_r2,rmse_mismatch,mse_mismatch,r2_mismatch,mae_mismatch
CNN,NaN,3734871.20,0.74,NaN,1255.74,3734871.20,1932.60,0.74,True,False,False,True
LSTM,1394.37,1646816.85,0.91,821.29,800.01,1646816.85,1283.77,0.91,True,False,False,True
MLP,1932.60,1750258.53,0.89,1255.74,835.54,1750258.53,1322.57,0.89,True,False,False,True
RNN,NaN,1945628.38,0.88,458.91,821.29,1945628.38,1394.37,0.88,True,False,False,True
Transformer,1283.77,1475395.99,0.96,800.01,760.48,1475395.99,1214.00,0.96,True,False,False,True


The table above separates initial extracted values from verified paper reference values. Mismatch columns highlight metrics that do not align with the PDF extraction, so final evaluation uses explicit `paper_*` values.


### Dataset Validation

Load `train.csv` and `store.csv`, merge them, and run initial checks: dataset size, date range, store count, missing values, duplicates, zero-sales rows, and closed-store rows.


In [4]:
def preprocess_rossmann(train_path: Path, store_path: Path) -> pd.DataFrame:
    train = pd.read_csv(train_path, low_memory=False)
    store = pd.read_csv(store_path, low_memory=False)
    data = train.merge(store, how='left', on='Store')
    data['Date'] = pd.to_datetime(data['Date'], errors='coerce')
    data = data.sort_values(['Date', 'Store']).reset_index(drop=True)

    data['MonthStr'] = data['Date'].dt.strftime('%b')

    def is_promo2_active(row):
        if row['Promo2'] == 1 and isinstance(row['PromoInterval'], str):
            return int(row['MonthStr'] in row['PromoInterval'].split(','))
        return 0

    data['IsPromo2Active'] = data.apply(is_promo2_active, axis=1)
    promo2_weeks = ((data['Date'].dt.year - data['Promo2SinceYear']) * 52 +
                    (data['Date'].dt.isocalendar().week.astype(float) - data['Promo2SinceWeek']))
    data['Promo2DurationWeeks'] = np.where(data['Promo2'] == 1, promo2_weeks, 0)
    data['Promo2DurationWeeks'] = data['Promo2DurationWeeks'].clip(lower=0).fillna(0)

    fill_zero_cols = ['CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear']
    data[fill_zero_cols] = data[fill_zero_cols].fillna(0)

    data = pd.get_dummies(
        data,
        columns=['StoreType', 'Assortment', 'StateHoliday'],
        drop_first=True,
    )
    bool_cols = data.select_dtypes(include='bool').columns
    data[bool_cols] = data[bool_cols].astype(int)
    data = data.drop(columns=['PromoInterval', 'MonthStr'], errors='ignore')
    data = data.dropna(subset=['Date', 'Sales'])
    return data


raw_df = preprocess_rossmann(TRAIN_PATH, STORE_PATH)
print(raw_df.shape)
print(raw_df['Date'].min(), raw_df['Date'].max())

(1017209, 24)
2013-01-01 00:00:00 2015-07-31 00:00:00


Merged dataset shape and date range confirm successful loading from the project data directory. This merged dataset is the base for temporal splitting, while `Sales` remains on the original scale.


### Modeling Data Preparation

Configuration:
- `OPEN_ONLY = False` — closed stores are retained.
- `POSITIVE_SALES_ONLY = False` — `Sales = 0` rows are retained.
- `MAX_ROWS = None` — full dataset is used.
- Split: time-ordered 80/20.
- Target `Sales` is not normalized; scaling is applied to input features only.


In [ ]:
POSITIVE_SALES_ONLY = False
MAX_ROWS = None
TEST_SIZE = 0.20

df = raw_df.copy()
df = df[df['Open'] == 1].copy()

if POSITIVE_SALES_ONLY:
    df = df[df['Sales'] > 0].copy()
if MAX_ROWS is not None:
    df = df.tail(MAX_ROWS).copy()

n_train = int(len(df) * (1 - TEST_SIZE))
train_df = df.iloc[:n_train].copy()
test_df = df.iloc[n_train:].copy()

target_col = 'Sales'
feature_cols = [c for c in train_df.columns if c not in [target_col, 'Date']]

X_train_raw = train_df[feature_cols].to_numpy(dtype=np.float32)
y_train = train_df[target_col].to_numpy(dtype=np.float32)
X_test_raw = test_df[feature_cols].to_numpy(dtype=np.float32)
y_test = test_df[target_col].to_numpy(dtype=np.float32)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_test = scaler.transform(X_test_raw).astype(np.float32)

# Safety for neural training: sklearn/TensorFlow must not receive NaN/Inf.
X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
y_train = np.nan_to_num(y_train, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
y_test = np.nan_to_num(y_test, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

assert np.isfinite(X_train).all(), 'X_train masih mengandung NaN/Inf'
assert np.isfinite(X_test).all(), 'X_test masih mengandung NaN/Inf'
assert np.isfinite(y_train).all(), 'y_train masih mengandung NaN/Inf'
assert np.isfinite(y_test).all(), 'y_test masih mengandung NaN/Inf'

print('rows:', len(df))
print('train:', len(train_df), 'test:', len(test_df), 'features:', len(feature_cols))
print('target normalized:', False)
print('X finite:', np.isfinite(X_train).all(), np.isfinite(X_test).all())
print('y range:', float(y_train.min()), float(y_train.max()))
pd.Series(y_train).describe()

rows: 1017209
train: 813767 test: 203442 features: 22
target normalized: False
X finite: True True
y range: 0.0 38037.0


count    813767.000000
mean       5739.765137
std        3845.520020
min           0.000000
25%        3686.000000
50%        5697.000000
75%        7811.000000
max       38037.000000
dtype: float64

Train/test size, feature count, and finite checks for `X`/`y` confirm the data is ready for neural-network training. `y_train` and `y_test` remain non-normalized so RMSE, MSE, R², and MAE are directly comparable to the paper.


## Modeling

Train Transformer, LSTM, MLP, RNN, and CNN on the reference preprocessing pipeline. Each model is evaluated on the held-out test set using RMSE, MSE, R², and MAE.


### Model Implementation

Shared building blocks and metric helpers for all neural-network baselines.


In [7]:
def dense_block(x, units: int, dropout: float):
    x = layers.Dense(units, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    return x


def transformer_encoder_block(x, d_model: int, num_heads: int, ffn_units: int, dropout: float):
    attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)(x, x)
    attn = layers.Dropout(dropout)(attn)
    x = layers.LayerNormalization(epsilon=1e-6)(layers.Add()([x, attn]))
    ffn = layers.Dense(ffn_units, activation='relu')(x)
    ffn = layers.Dropout(dropout)(ffn)
    ffn = layers.Dense(d_model)(ffn)
    return layers.LayerNormalization(epsilon=1e-6)(layers.Add()([x, ffn]))


def output_regression_layer(output_bias=None):
    if output_bias is None:
        return layers.Dense(1)
    return layers.Dense(1, bias_initializer=keras.initializers.Constant(output_bias))


def build_model(model_name: str, n_features: int, learning_rate=1e-4, dropout=0.10, output_bias=None):
    if model_name == 'MLP':
        inputs = keras.Input(shape=(n_features,))
        x = dense_block(inputs, 1024, dropout)
        x = dense_block(x, 768, dropout)
        x = dense_block(x, 512, dropout)
        x = dense_block(x, 256, dropout)
        x = dense_block(x, 128, dropout)
        x = dense_block(x, 64, dropout)
        outputs = output_regression_layer(output_bias)(x)
    else:
        inputs = keras.Input(shape=(1, n_features))
        if model_name == 'LSTM':
            x = layers.LSTM(256, return_sequences=True, dropout=dropout)(inputs)
            x = layers.LSTM(128, return_sequences=True, dropout=dropout)(x)
            x = layers.LSTM(64, dropout=dropout)(x)
        elif model_name == 'RNN':
            x = layers.SimpleRNN(256, return_sequences=True, dropout=dropout)(inputs)
            x = layers.SimpleRNN(128, return_sequences=True, dropout=dropout)(x)
            x = layers.SimpleRNN(64, dropout=dropout)(x)
        elif model_name == 'CNN':
            x = layers.Conv1D(256, kernel_size=1, activation='relu')(inputs)
            x = layers.BatchNormalization()(x)
            x = layers.Conv1D(256, kernel_size=1, activation='relu')(x)
            x = layers.BatchNormalization()(x)
            x = layers.Conv1D(128, kernel_size=1, activation='relu')(x)
            x = layers.BatchNormalization()(x)
            x = layers.Conv1D(64, kernel_size=1, activation='relu')(x)
            x = layers.BatchNormalization()(x)
            x = layers.Flatten()(x)
        elif model_name == 'Transformer':
            d_model = 256
            num_heads = 8
            x = layers.Dense(d_model)(inputs)
            x = transformer_encoder_block(x, d_model=d_model, num_heads=num_heads, ffn_units=512, dropout=dropout)
            x = transformer_encoder_block(x, d_model=d_model, num_heads=num_heads, ffn_units=512, dropout=dropout)
            x = transformer_encoder_block(x, d_model=d_model, num_heads=num_heads, ffn_units=256, dropout=dropout)
            x = transformer_encoder_block(x, d_model=d_model, num_heads=num_heads, ffn_units=256, dropout=dropout)
            x = layers.GlobalAveragePooling1D()(x)
        else:
            raise ValueError(model_name)

        x = dense_block(x, 512, dropout)
        x = dense_block(x, 256, dropout)
        x = dense_block(x, 128, dropout)
        x = dense_block(x, 64, dropout)
        outputs = output_regression_layer(output_bias)(x)

    model = keras.Model(inputs, outputs, name=f'diamantini_rossmann_{model_name.lower()}')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate, clipnorm=0.5),
        loss='mse',
        metrics=['mae'],
    )
    return model


def compute_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        'rmse': float(np.sqrt(mse)),
        'mse': float(mse),
        'r2': float(r2_score(y_true, y_pred)),
        'mae': float(mean_absolute_error(y_true, y_pred)),
    }

#### Training

Run all models sequentially. TensorFlow automatically uses **GPU when available** and falls back to **CPU** otherwise. Batch size is auto-selected per device (`1024` on GPU, `256` on CPU). Results are collected in `repro_df`.


In [ ]:
MODELS_TO_RUN = ['Transformer', 'LSTM', 'MLP', 'RNN', 'CNN']
EPOCHS = 100
BATCH_SIZE = 1024 if GPU_AVAILABLE else 256
VALIDATION_SPLIT = 0.10
PATIENCE = 12
OUTPUT_BIAS = float(np.mean(y_train))
MODEL_LEARNING_RATES = {
    'Transformer': 1e-4,
    'LSTM': 3e-4,
    'MLP': 5e-4,
    'RNN': 2e-4,
    'CNN': 5e-4,
}

X_train_seq = X_train[:, np.newaxis, :]
X_test_seq = X_test[:, np.newaxis, :]

print('=' * 60)
print('Training configuration')
print('=' * 60)
if GPU_AVAILABLE:
    print(f'[INFO] Using GPU for deep learning training ({DEVICE_NAME}).')
else:
    print('[WARN] GPU not available — falling back to CPU training.')
    print('[WARN] Expect significantly longer runtime on the full Rossmann dataset.')
print(f'Compute device : {COMPUTE_DEVICE}')
print(f'Device name    : {DEVICE_NAME}')
print(f'Batch size     : {BATCH_SIZE} (auto-selected for {COMPUTE_DEVICE})')
print(f'Output bias    : {OUTPUT_BIAS:.4f}')
print(f'Models to run  : {MODELS_TO_RUN}')
print('=' * 60)

repro_rows = []
histories = {}

for model_name in MODELS_TO_RUN:
    print(f'\n=== {model_name} ===')
    print(f'Compute device : {COMPUTE_DEVICE} ({DEVICE_NAME})')
    print('Learning rate  :', MODEL_LEARNING_RATES[model_name])
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)

    with TRAINING_STRATEGY.scope():
        model = build_model(
            model_name,
            n_features=X_train.shape[1],
            learning_rate=MODEL_LEARNING_RATES[model_name],
            output_bias=OUTPUT_BIAS,
        )

    X_fit = X_train if model_name == 'MLP' else X_train_seq
    X_eval = X_test if model_name == 'MLP' else X_test_seq

    callbacks = [
        keras.callbacks.TerminateOnNaN(),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5),
    ]
    history = model.fit(
        X_fit,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=VALIDATION_SPLIT,
        shuffle=False,
        callbacks=callbacks,
        verbose=1,
    )

    y_pred = model.predict(X_eval, batch_size=BATCH_SIZE, verbose=0).reshape(-1)
    y_pred = np.nan_to_num(y_pred, nan=np.nanmean(y_train), posinf=np.nanmean(y_train), neginf=0.0)
    metrics = compute_metrics(y_test, y_pred)
    repro_rows.append({'model': model_name, 'epochs_ran': len(history.history['loss']), **metrics})
    histories[model_name] = history.history
    print(metrics)

repro_df = pd.DataFrame(repro_rows)
repro_df

Training configuration
[WARN] GPU not available — falling back to CPU training.
[WARN] Expect significantly longer runtime on the full Rossmann dataset.
Compute device : CPU
Device name    : /device:CPU:0
Batch size     : 256 (auto-selected for CPU)
Output bias    : 5739.7651
Models to run  : ['Transformer', 'LSTM', 'MLP', 'RNN', 'CNN']

=== Transformer ===
Compute device : CPU (/device:CPU:0)
Learning rate  : 0.0001

Epoch 1/100
 219/2861 ━━━━━━━━━━━━━━━━━━━━ 21:30 489ms/step - loss: 13872837.0000 - mae: 2857.8042

Training output aggregates Transformer, LSTM, MLP, RNN, and CNN performance in `repro_df`. `epochs_ran` reflects early stopping, while RMSE, MSE, R², and MAE are the primary comparison metrics against the paper.


## Reference Results

Compute RMSE, MSE, R², and MAE on the test set and compare reproduction results against `PAPER_REFERENCE`. Delta and percentage-difference columns show the gap between reproduction and paper values.


In [ ]:
paper_df = pd.DataFrame(PAPER_REFERENCE).T.reset_index().rename(columns={'index': 'model'})
comparison = paper_df.merge(repro_df, on='model', suffixes=('_paper', '_reproduced'))

for metric in ['rmse', 'mse', 'r2', 'mae']:
    comparison[f'delta_{metric}'] = comparison[f'{metric}_reproduced'] - comparison[f'{metric}_paper']
    comparison[f'pct_diff_{metric}'] = np.where(
        comparison[f'{metric}_paper'] != 0,
        comparison[f'delta_{metric}'] / comparison[f'{metric}_paper'] * 100,
        np.nan,
    )

comparison[[
    'model',
    'rmse_paper', 'rmse_reproduced', 'delta_rmse', 'pct_diff_rmse',
    'mse_paper', 'mse_reproduced', 'delta_mse', 'pct_diff_mse',
    'r2_paper', 'r2_reproduced', 'delta_r2',
    'mae_paper', 'mae_reproduced', 'delta_mae', 'pct_diff_mae',
    'epochs_ran',
]]

The comparison table is the main reproduction summary: each metric shows paper value, reproduced value, absolute delta, and percentage difference. Smaller `delta_*` or `pct_diff_*` indicates closer alignment; large gaps should be read together with architecture/preprocessing deviations and PDF extraction inconsistencies.


## Best Baseline & Export (Reference)


In [ ]:
best_reference = repro_df.sort_values(['rmse', 'mae']).iloc[0]
df_compare_ref = repro_df[['model', 'rmse', 'mse', 'r2', 'mae', 'epochs_ran']].copy()
df_compare_ref = df_compare_ref.rename(
    columns={
        'rmse': 'Reference RMSE',
        'mse': 'Reference MSE',
        'r2': 'Reference R2',
        'mae': 'Reference MAE',
        'epochs_ran': 'Epochs Ran',
    }
)
print('Best Baseline (Reference Preprocessing):')
display(best_reference)
display(df_compare_ref.sort_values('Reference RMSE'))


# Exploration Based on Our Preprocessing

This section applies the preprocessing pipeline from `our_study_rosman.ipynb`, then trains the **same deep learning models** as the reference section (Transformer, LSTM, MLP, RNN, CNN) for a direct comparison of preprocessing impact.


## Preprocessing

Pipeline aligned with `our_study_rosman.ipynb`:
- Merge `train.csv` + `store.csv`
- Create `IsPromo2Active` and `Promo2DurationWeeks`
- One-hot encode `StoreType`, `Assortment`, `StateHoliday`
- Sort columns (target `Sales` last)
- Drop competition columns and redundant promo metadata
- Keep `Date` for visualization only (excluded from model features)


In [ ]:
# Load and merge Rossmann data (same base load as our_study_rosman)
train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
store_df = pd.read_csv(STORE_PATH, low_memory=False)
data = pd.merge(train_df, store_df, how='left', on='Store')
data['Date'] = pd.to_datetime(data['Date'], errors='coerce')
series = data.sort_values('Date').reset_index(drop=True)

# --- Feature engineering (our_study_rosman) ---
series['MonthStr'] = series['Date'].dt.strftime('%b')


def is_promo2_active(row):
    if row['Promo2'] == 1 and isinstance(row['PromoInterval'], str):
        return 1 if row['MonthStr'] in row['PromoInterval'].split(',') else 0
    return 0


series['IsPromo2Active'] = series.apply(is_promo2_active, axis=1)
series['Promo2DurationWeeks'] = np.where(
    series['Promo2'] == 1,
    (series['Date'].dt.year - series['Promo2SinceYear']) * 52
    + (series['Date'].dt.isocalendar().week - series['Promo2SinceWeek']),
    0,
)
series['Promo2DurationWeeks'] = series['Promo2DurationWeeks'].clip(lower=0).fillna(0)

cat_cols = ['StoreType', 'Assortment', 'StateHoliday']
series = pd.get_dummies(series, columns=cat_cols, drop_first=True)
boolean_cols = series.select_dtypes(include='bool').columns
series[boolean_cols] = series[boolean_cols].astype(int)

target_col = 'Sales'
feature_order = sorted([c for c in series.columns if c != target_col]) + [target_col]
series = series[feature_order]

series.drop(
    columns=[
        'CompetitionDistance',
        'CompetitionOpenSinceMonth',
        'CompetitionOpenSinceYear',
        'Promo2SinceWeek',
        'Promo2SinceYear',
        'PromoInterval',
        'MonthStr',
    ],
    inplace=True,
    errors='ignore',
)
series.dropna(inplace=True)

print('Our preprocessing complete.')
print('Shape:', series.shape)
print('Columns:', list(series.columns))
series.head()

### Modeling Data Preparation

Same modeling setup as the reference section, but using the `our_study_rosman` feature set:
- Chronological 80/20 split
- Raw `Sales` target (non-normalized)
- `StandardScaler` on input features only


In [ ]:
POSITIVE_SALES_ONLY_OUR = False
MAX_ROWS_OUR = None
TEST_SIZE_OUR = 0.20

df_our = series.copy()
if POSITIVE_SALES_ONLY_OUR:
    df_our = df_our[df_our['Sales'] > 0].copy()
if MAX_ROWS_OUR is not None:
    df_our = df_our.tail(MAX_ROWS_OUR).copy()

n_train_our = int(len(df_our) * (1 - TEST_SIZE_OUR))
train_df_our = df_our.iloc[:n_train_our].copy()
test_df_our = df_our.iloc[n_train_our:].copy()

target_col_our = 'Sales'
feature_cols_our = [c for c in train_df_our.columns if c not in [target_col_our, 'Date']]

X_train_raw_our = train_df_our[feature_cols_our].to_numpy(dtype=np.float32)
y_train_our = train_df_our[target_col_our].to_numpy(dtype=np.float32)
X_test_raw_our = test_df_our[feature_cols_our].to_numpy(dtype=np.float32)
y_test_our = test_df_our[target_col_our].to_numpy(dtype=np.float32)

scaler_our = StandardScaler()
X_train_our = scaler_our.fit_transform(X_train_raw_our).astype(np.float32)
X_test_our = scaler_our.transform(X_test_raw_our).astype(np.float32)

X_train_our = np.nan_to_num(X_train_our, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
X_test_our = np.nan_to_num(X_test_our, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
y_train_our = np.nan_to_num(y_train_our, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
y_test_our = np.nan_to_num(y_test_our, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

assert np.isfinite(X_train_our).all(), 'X_train_our contains NaN/Inf'
assert np.isfinite(X_test_our).all(), 'X_test_our contains NaN/Inf'
assert np.isfinite(y_train_our).all(), 'y_train_our contains NaN/Inf'
assert np.isfinite(y_test_our).all(), 'y_test_our contains NaN/Inf'

print('rows:', len(df_our))
print('train:', len(train_df_our), 'test:', len(test_df_our), 'features:', len(feature_cols_our))
print('target normalized:', False)
pd.Series(y_train_our).describe()

## Modeling

Train the same deep learning models as the reference section (Transformer, LSTM, MLP, RNN, CNN) on the our-preprocessing feature set.

#### Training


In [ ]:
MODELS_TO_RUN_OUR = ['Transformer', 'LSTM', 'MLP', 'RNN', 'CNN']
EPOCHS_OUR = 100
BATCH_SIZE_OUR = 1024 if GPU_AVAILABLE else 256
VALIDATION_SPLIT_OUR = 0.10
PATIENCE_OUR = 12
OUTPUT_BIAS_OUR = float(np.mean(y_train_our))
MODEL_LEARNING_RATES_OUR = {
    'Transformer': 1e-4,
    'LSTM': 3e-4,
    'MLP': 5e-4,
    'RNN': 2e-4,
    'CNN': 5e-4,
}

X_train_seq_our = X_train_our[:, np.newaxis, :]
X_test_seq_our = X_test_our[:, np.newaxis, :]

print('=' * 60)
print('Our preprocessing — training configuration')
print('=' * 60)
if GPU_AVAILABLE:
    print(f'[INFO] Using GPU for deep learning training ({DEVICE_NAME}).')
else:
    print('[WARN] GPU not available — falling back to CPU training.')
print(f'Compute device : {COMPUTE_DEVICE}')
print(f'Batch size     : {BATCH_SIZE_OUR} (auto-selected for {COMPUTE_DEVICE})')
print(f'Output bias    : {OUTPUT_BIAS_OUR:.4f}')
print(f'Models to run  : {MODELS_TO_RUN_OUR}')
print('=' * 60)

repro_rows_our = []
histories_our = {}

for model_name in MODELS_TO_RUN_OUR:
    print(f'\n=== {model_name} (Our Preprocessing) ===')
    print(f'Compute device : {COMPUTE_DEVICE} ({DEVICE_NAME})')
    print('Learning rate  :', MODEL_LEARNING_RATES_OUR[model_name])
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)

    with TRAINING_STRATEGY.scope():
        model = build_model(
            model_name,
            n_features=X_train_our.shape[1],
            learning_rate=MODEL_LEARNING_RATES_OUR[model_name],
            output_bias=OUTPUT_BIAS_OUR,
        )

    X_fit = X_train_our if model_name == 'MLP' else X_train_seq_our
    X_eval = X_test_our if model_name == 'MLP' else X_test_seq_our

    callbacks = [
        keras.callbacks.TerminateOnNaN(),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE_OUR, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5),
    ]
    history = model.fit(
        X_fit,
        y_train_our,
        epochs=EPOCHS_OUR,
        batch_size=BATCH_SIZE_OUR,
        validation_split=VALIDATION_SPLIT_OUR,
        shuffle=False,
        callbacks=callbacks,
        verbose=1,
    )

    y_pred = model.predict(X_eval, batch_size=BATCH_SIZE_OUR, verbose=0).reshape(-1)
    y_pred = np.nan_to_num(y_pred, nan=np.nanmean(y_train_our), posinf=np.nanmean(y_train_our), neginf=0.0)
    metrics = compute_metrics(y_test_our, y_pred)
    repro_rows_our.append({'model': model_name, 'epochs_ran': len(history.history['loss']), **metrics})
    histories_our[model_name] = history.history
    print(metrics)

repro_df_our = pd.DataFrame(repro_rows_our)
repro_df_our

## Our Results

Results from deep learning models trained on the `our_study_rosman` preprocessing pipeline.


## Best Baseline & Export (Our Preprocessing)


In [ ]:
best_our = repro_df_our.sort_values(['rmse', 'mae']).iloc[0]
df_compare_our = repro_df_our[['model', 'rmse', 'mse', 'r2', 'mae', 'epochs_ran']].copy()
df_compare_our = df_compare_our.rename(
    columns={
        'rmse': 'Our Preprocessing RMSE',
        'mse': 'Our Preprocessing MSE',
        'r2': 'Our Preprocessing R2',
        'mae': 'Our Preprocessing MAE',
        'epochs_ran': 'Epochs Ran',
    }
)

print('Best Baseline (Our Preprocessing):')
display(best_our)
display(df_compare_our.sort_values('Our Preprocessing RMSE'))

# Summary
